# Customer Support — Satisfaction Prediction

## Step 1: Problem Definition
Predict **customer satisfaction class** (Low / Mid / High) from ticket and service features.

In [ ]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

sns.set_theme(style='whitegrid')
print('Project root:', PROJECT_ROOT)

## Step 2: Data Collection

In [ ]:
from src.data_loader import load_tickets
raw_df = load_tickets(use_db=True)
raw_df.head()

## Step 3: Data Cleaning

In [ ]:
from src.preprocessor import clean_data
cleaned_df = clean_data(raw_df, task_type='satisfaction')
cleaned_df['customer_satisfaction_score'].value_counts().sort_index()

### EDA: Satisfaction by channel

In [ ]:
import seaborn as sns
sns.boxplot(data=cleaned_df, x='channel', y='customer_satisfaction_score')
plt.xticks(rotation=45)
plt.title('Satisfaction by Channel')
plt.show()

## Step 4: Feature Engineering
Map 1–5 score to Low / Mid / High classes.

In [ ]:
from src.preprocessor import engineer_features
from src.label_engineering import derive_satisfaction_class

fe_df = engineer_features(cleaned_df, task_type='satisfaction')
fe_df['satisfaction_class'] = derive_satisfaction_class(fe_df)
fe_df = fe_df.sample(min(30000, len(fe_df)), random_state=42)
fe_df['satisfaction_class'].value_counts()

## Step 5: Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = fe_df.drop(columns=['customer_satisfaction_score', 'satisfaction_class'])
for col in X.select_dtypes(include=['object']).columns:
    X[col] = X[col].fillna('Unknown')
y = fe_df['satisfaction_class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape)

## Step 6: Model Selection

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from src.model_trainer import get_satisfaction_pipelines

for name, pipe in get_satisfaction_pipelines(X_train).items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    print(f"{name}: Acc={accuracy_score(y_test, pred):.4f}  F1={f1_score(y_test, pred, average='weighted'):.4f}")

## Step 7: Model Training

In [ ]:
sat = get_satisfaction_pipelines(X_train)['Random Forest']
sat.fit(X_train, y_train)

## Step 8: Model Evaluation

In [ ]:
from sklearn.metrics import classification_report

y_pred = sat.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

## Step 9: Model Tuning

In [ ]:
print('Optional GridSearchCV for satisfaction model.')

## Step 10: Deployment

In [ ]:
import joblib
from src.paths import get_models_dir

acc = accuracy_score(y_test, y_pred)
joblib.dump({'model': sat, 'model_name': 'Random Forest', 'accuracy': acc},
            os.path.join(get_models_dir(), 'satisfaction_model.pkl'))
print('Saved satisfaction model. Accuracy=', acc)